# Phase 4 — Statistical Analysis

This phase summarizes central tendency and spread, identifies IQR outliers for investigation, measures monotonic relationships with Spearman correlation, and compares rating distributions across high- and low-discount products with a Mann–Whitney U test. Ratings are bounded and highly concentrated near the upper end, so a rank-based group comparison is a defensible choice. The test is associational and does not show that discounts cause ratings to change.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
Path("phase_outputs").mkdir(exist_ok=True)
from scipy.stats import spearmanr, mannwhitneyu

df = pd.read_csv("../electronics_analysis_ready.csv")
num = ["discount_price", "actual_price", "review_rating", "no_of_ratings", "discount_percent"]
num = [c for c in num if c in df]
summary = df[num].describe().T
summary["median"] = df[num].median()
summary["mode"] = df[num].mode(dropna=True).iloc[0]
summary["variance"] = df[num].var()
summary["standard_deviation"] = df[num].std()
summary["skewness"] = df[num].skew()
summary["IQR"] = df[num].quantile(.75) - df[num].quantile(.25)
display(summary.round(3))

outliers=[]
for col in num:
    values=df[col].dropna(); q1=values.quantile(.25); q3=values.quantile(.75); iqr=q3-q1
    lower=q1-1.5*iqr; upper=q3+1.5*iqr
    outliers.append({"feature":col,"lower_fence":lower,"upper_fence":upper,"outlier_count":int(((values<lower)|(values>upper)).sum()),"outlier_pct":float(((values<lower)|(values>upper)).mean()*100)})
outlier_report=pd.DataFrame(outliers)
display(outlier_report.round(3))
print("IQR flags identify values for investigation; they are not automatic deletion rules.")

correlations=[]
for a,b in [("discount_price","review_rating"),("no_of_ratings","review_rating"),("discount_percent","review_rating"),("actual_price","discount_price")]:
    pair=df[[a,b]].dropna()
    rho,p=spearmanr(pair[a],pair[b])
    correlations.append({"feature_1":a,"feature_2":b,"n":len(pair),"spearman_rho":rho,"p_value":p})
correlation_report=pd.DataFrame(correlations)
display(correlation_report.round(4))

,count,mean,std,min,25%,50%,75%,max,median,mode,variance,standard_deviation,skewness,IQR
discount_price,4740.0,3078.191,8564.866,36.000,299.000,649.000,1699.000,127999.000,649.000,299.00,7.335693e+07,8564.866,6.735,1400.000
actual_price,4979.0,5226.882,11861.096,35.000,835.000,1499.000,3999.000,149999.000,1499.000,999.00,1.406856e+08,11861.096,5.600,3164.000
review_rating,5010.0,4.488,1.086,1.000,4.000,5.000,5.000,5.000,5.000,5.00,1.179000e+00,1.086,-2.334,1.000
no_of_ratings,4958.0,13260.089,34803.226,1.000,509.250,2321.000,9812.750,437651.000,2321.000,1.00,1.211265e+09,34803.226,6.083,9303.500
discount_percent,4740.0,51.882,21.985,0.004,35.035,55.576,70.023,97.889,55.576,70.07,4.833600e+02,21.985,-0.425,34.988


,feature,lower_fence,upper_fence,outlier_count,outlier_pct
0,discount_price,-1801.000,3799.000,633,13.354
1,actual_price,-3911.000,8745.000,632,12.693
2,review_rating,2.500,6.500,406,8.104
3,no_of_ratings,-13446.000,23768.000,681,13.735
4,discount_percent,-17.447,122.506,0,0.000


IQR flags identify values for investigation; they are not automatic deletion rules.


,feature_1,feature_2,n,spearman_rho,p_value
0,discount_price,review_rating,4740,-0.0147,0.3118
1,no_of_ratings,review_rating,4958,-0.0855,0.0000
2,discount_percent,review_rating,4740,0.0177,0.2236
3,actual_price,discount_price,4740,0.9049,0.0000


## Discount group comparison

The null hypothesis states that rating distributions are the same in the two discount groups. Products are split at the median observed discount. Significance level: alpha = 0.05 (two-sided). We report the group medians, Mann–Whitney U statistic, p-value, and rank-biserial effect size. The p-value is computed from the current data every time the notebook runs.

In [2]:
from scipy.stats import mannwhitneyu
valid=df[["discount_percent","review_rating"]].dropna()
median_discount=valid.discount_percent.median()
high=valid.loc[valid.discount_percent>median_discount,"review_rating"]
low=valid.loc[valid.discount_percent<=median_discount,"review_rating"]
u_stat,p_value=mannwhitneyu(high,low,alternative="two-sided")
# Rank-biserial r = 2U/(n1*n2) - 1, where U belongs to the high-discount group.
# r > 0 means high-discount rows tend to have higher ratings; r < 0 means lower.
effect=2*u_stat/(len(high)*len(low))-1
print("High-discount group n / median rating:",len(high),round(high.median(),3))
print("Low-or-equal-discount group n / median rating:",len(low),round(low.median(),3))
print("Mann–Whitney U:",round(float(u_stat),3))
print("Two-sided p-value:",float(p_value))
print("Rank-biserial effect size (positive = high-discount group rated higher):",round(float(effect),4))
alpha=.05
if p_value<alpha:
    print("Decision: reject H0 at alpha=0.05; the observed rating distributions differ.")
else:
    print("Decision: fail to reject H0 at alpha=0.05; evidence is insufficient to conclude that the distributions differ.")
print("This comparison is observational and does not establish causation.")

plt.figure(figsize=(8,5))
plt.boxplot([low,high],tick_labels=["At or below median discount","Above median discount"])
plt.ylabel("Review rating (out of 5)")
plt.title("Rating distributions by discount group")
plt.tight_layout()
plt.savefig("phase_outputs/phase4_discount_rating_groups.png", dpi=180, bbox_inches="tight")
plt.show()

High-discount group n / median rating: 2370 5.0
Low-or-equal-discount group n / median rating: 2370 5.0
Mann–Whitney U: 2833249.5
Two-sided p-value: 0.4906660325993484
Rank-biserial effect size (positive = high-discount group rated higher): 0.0088
Decision: fail to reject H0 at alpha=0.05; evidence is insufficient to conclude that the distributions differ.
This comparison is observational and does not establish causation.


/var/folders/_y/jrtd4g7n1xq8nyb6qbq1yywh0000gn/T/ipykernel_98237/2458348912.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Interpretation

A non-significant result does not prove that the groups are identical. A small effect can also be difficult to detect when ratings cluster near 4–5 stars. Review-count and price associations are descriptive and may be affected by product category, repeated listings, and the age of the historical dataset.